# レッスン 10 - 本番環境での AI エージェント

このレッスンでは、Microsoft Agent Framework (Python) を使用した AI エージェントのための **プロダクションパターン** を学びます。内容は次のとおりです：

- **可観測性** — エージェントのやり取りにタイミングとログ記録を追加する
- **評価** — 応答の品質を採点するために評価エージェントを使用する
- **コスト管理** — トークン最適化とモデル選択の戦略

シナリオは、ユーザーの旅行計画を支援する **旅行代理店** で、監視と評価がその上に重ねられています。


## セットアップ


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## 本番環境での考慮事項

AIエージェントをプロトタイプから本番に移行するには、次の3つの柱に細心の注意を払う必要があります：

1. **Observability** — エージェントが何をしているか、どれくらい時間がかかっているか、どのツールを呼び出しているかを可視化する必要があります。トレースやログがなければ、本番環境の問題をデバッグすることはほとんど不可能です。

2. **Evaluation** — 自動化された品質チェックは、エージェントの応答が時間経過とともに正確で完全かつ有用であることを保証します。評価エージェントは、定義された基準に対して応答にスコアを付けることができます。

3. **Cost Management** — トークン使用量は直接コストに影響します。プロンプト最適化、モデル選択、キャッシュなどの戦略は、品質を犠牲にすることなく費用を抑えるのに役立ちます。


## 観測可能なエージェントの構築

トラベル用のツールを定義し、エージェントの呼び出しを時間計測でラップしてレイテンシを監視できるようにします。 本番環境では OpenTelemetry や同様のトレーシングバックエンドと統合します。


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## 評価パターン

一般的な運用パターンは、第二のエージェントを**評価者**として使用することです。評価者は、一次エージェントの応答を、完全性、正確性、有用性などの事前定義された基準に照らして採点します。

これにより:
- ユーザーに届く前の応答に対する自動品質ゲート
- プロンプトやモデルが変更されたときの回帰検出
- 時間経過に伴うエージェントの性能の継続的な監視


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## コスト管理戦略

コスト管理は本番環境のAIエージェントにとって重要です。以下は主要な戦略です：

| 戦略 | 説明 |
|---|---|
| **プロンプト最適化** | システム指示は簡潔に保ちます。冗長なコンテキストを削除して入力トークンを減らします。 |
| **モデル選択** | 分類や抽出のような単純なタスクには、より小さく安価なモデル（例: GPT-4o-mini）を使用し、複雑な推論にはより高性能なモデルを割り当てます。 |
| **キャッシュ** | ツール結果や頻繁なクエリをキャッシュして、冗長なAPI呼び出しを避けます。 |
| **トークン予算** | `max_tokens` の上限を設定して、予期せず長くなる応答を防ぎます。 |
| **バッチ処理** | 可能な場合は複数のユーザークエリを1つのAPI呼び出しにまとめます。 |

実際には、階層化されたアプローチが有効です：単純なリクエストは高速で安価なモデルにルーティングし、複雑なクエリのみをより高性能なモデルにエスカレーションします。


## Summary

このレッスンで学んだことは次のとおりです：

1. **可観測性を追加する** エージェントのやり取りに対してタイミングやログを用いることで、トレーシングとモニタリングの基盤を築くこと。
2. **エージェントの応答を評価する** 完全性、正確性、有用性をスコア化する評価エージェントを使って自動的に評価すること。
3. **コストを管理する** プロンプト最適化、モデル選択、キャッシュ、トークン予算を通じてコストを管理すること。

これらの本番運用パターンは、AIエージェントが大規模において信頼性があり、測定可能で、コスト効率が高くなることを保証するのに役立ちます。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
免責事項：
この文書は AI 翻訳サービス「[Co-op Translator](https://github.com/Azure/co-op-translator)」を使用して翻訳されました。正確性には努めていますが、自動翻訳には誤りや不正確な表現が含まれる可能性があることをご承知ください。原文（原語）の文書が正式な出典とみなされます。重要な情報については、専門の翻訳者による人手翻訳を推奨します。本翻訳の利用により生じた誤解や解釈の相違について、当方は一切責任を負いません。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
